In [ ]:
import pandas as pd
import os 
import sys
import numpy as np
np.random.seed(42)

real_df = pd.read_csv('../data/private/clean_data.csv')
synthetic_df = pd.read_csv('../data/synthetic/CopulaGAN.csv')




In [ ]:
from sdmetrics.single_table import CategoricalKNN, CategoricalRF, CategoricalNB
key_fields = [
    'Age at Diagnosis',
    'Sex',
    'BMI',
    'Cranial Location (Infratentorial, supratentorial)',
    'WHO Grade',
    'Age at Sx'
]


CategoricalNB.compute(
    real_data=real_df,
    synthetic_data=synthetic_df,
    key_fields=key_fields,
    sensitive_fields=['MRN']
)

In [ ]:
# --- 0) Build `synthetic_datasets` from CSVs in synthetic_data/ ---
import os
import pandas as pd
import numpy as np

os.makedirs("../legacy/original_submission/figures", exist_ok=True)

SYN_DIR = "../data/synthetic"
MODEL_FILES = {
    "GaussianCopula": "GaussianCopula_withanon.csv",
    "CopulaGAN": "CopulaGAN_withanon.csv",
    "CTGAN": "CTGAN_withanon.csv",
    "TVAE": "TVAE_withanon.csv"
}

# If SENSITIVE wasn't defined earlier, default to MRN to match your prior NB test
try:
    SENSITIVE
except NameError:
    SENSITIVE = ["MRN", "Date of Sx", "Initials", "Date of Birth"]

# Normalize SENSITIVE to a list to avoid list+str TypeError
if isinstance(SENSITIVE, str) or SENSITIVE is None:
    SENSITIVE = [SENSITIVE] if isinstance(SENSITIVE, str) else ["MRN"]


# Safety: ensure `key_fields` and `real_df` exist (they should from your earlier cells)
assert 'real_df' in globals(), "real_df not found. Make sure you ran the cell that loads clean_data.csv."
assert 'key_fields' in globals(), "key_fields not found. Run the cell where you defined key_fields."

synthetic_datasets = {}
for name, fname in MODEL_FILES.items():
    fpath = os.path.join(SYN_DIR, fname)
    if not os.path.exists(fpath):
        print(f"❌ Missing synthetic file for {name}: {fpath}")
        continue

    sdf = pd.read_csv(fpath)

    # Align dtypes to real_df for overlapping columns
    common = list(set(sdf.columns).intersection(set(real_df.columns)))
    for col in common:
        if pd.api.types.is_numeric_dtype(real_df[col]):
            sdf[col] = pd.to_numeric(sdf[col], errors='coerce')
        else:
            sdf[col] = sdf[col].astype(str)

    # Warn if attacker-required columns are missing
    needed = set(key_fields + SENSITIVE)
    missing = needed - set(sdf.columns)
    if missing:
        print(f"⚠️ {name} is missing attacker-required columns: {sorted(missing)}")

    synthetic_datasets[name] = sdf

print(f"✅ Loaded synthetic_datasets: {list(synthetic_datasets.keys())}")


In [ ]:
# --- 1) Run Attribute-Inference attackers across all synthetic datasets ---
from sdmetrics.single_table import CategoricalNB, CategoricalKNN, CategoricalRF
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

ATTACKERS = {
    'Naive Bayes': CategoricalNB,
    'KNN': CategoricalKNN,
    'Random Forest': CategoricalRF
}

attack_rows = []
for model_name, synth_df in synthetic_datasets.items():
    for atk_name, atk_cls in ATTACKERS.items():
        score = atk_cls.compute(
            real_data=real_df,
            synthetic_data=synth_df,
            key_fields=key_fields,
            sensitive_fields=SENSITIVE
        )
        attack_rows.append({
            'Model': model_name,
            'Attacker': atk_name,
            'Privacy Score (↑ better)': score
        })

attack_df = pd.DataFrame(attack_rows).sort_values(['Attacker','Model']).reset_index(drop=True)
attack_df


In [ ]:
# --- 2) Grouped bar + heatmap (poster-ready) ---
plt.figure(figsize=(10, 6), dpi=300)
pivot_df = attack_df.pivot(index='Model', columns='Attacker', values='Privacy Score (↑ better)').reindex(
    ['GaussianCopula','CopulaGAN','CTGAN','TVAE']
)
pivot_df.plot(kind='bar')
plt.title('Attribute-Inference Privacy Scores (Higher = Safer)')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../legacy/original_submission/figures/privacy_grouped_bar.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 4), dpi=300)
sns.heatmap(pivot_df, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar_kws={'label': 'Privacy Score'})
plt.title('Simulated Attribute-Inference Attacks on Synthetic Data')
plt.ylabel('Model')
plt.xlabel('Attacker')
plt.tight_layout()
plt.savefig('../legacy/original_submission/figures/privacy_heatmap1.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# --- 3) Privacy–Utility trade-off (uses your saved QualityReport PKLs or df_scores fallback) ---
from sdmetrics.reports.single_table import QualityReport

quality_overall = {}

# Try to load QualityReport pickles you saved earlier
for name in ['GaussianCopula','CopulaGAN','CTGAN','TVAE']:
    pkl_path = f'../legacy/original_submission/quality_analysis/quality_report_{name}.pkl'
    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            rpt = pickle.load(f)
        try:
            quality_overall[name] = rpt.get_score() * 100.0
        except:
            pass

# Fallback to df_scores (from your earlier cell) if present
if ('df_scores' in globals()):
    for _, row in df_scores.iterrows():
        quality_overall[row['Model']] = quality_overall.get(row['Model'], row['Overall'])

# Compute average privacy per model
avg_privacy = attack_df.groupby('Model')['Privacy Score (↑ better)'].mean().to_dict()

trade_rows = []
for name in ['GaussianCopula','CopulaGAN','CTGAN','TVAE']:
    if name in quality_overall and name in avg_privacy:
        trade_rows.append({
            'Model': name,
            'Utility (Overall Quality, %)': quality_overall[name],
            'Avg Privacy Score (↑ safer)': avg_privacy[name],
            'Privacy Risk (↓ better)': (1.0 - avg_privacy[name]) * 100.0
        })

trade_df = pd.DataFrame(trade_rows).sort_values('Model').reset_index(drop=True)
trade_df


In [ ]:
# --- Attribute-inference with clinically sensitive fields + MRN, poster-ready figs (DPI=300, risk-aware colors) ---

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

os.makedirs("../legacy/original_submission/figures", exist_ok=True)

# 1) Include MRN explicitly (keeps your previous candidates)
ALT_SENSITIVE_CANDIDATES = [
    'MRN',
    'Initials',
    'Date of Birth',
    'Date of Sx',
    'Death (Y/N)',
    'Recurrence',
    'Postop complication',
    'Age at Diagnosis',
    'BMI',
    'Cranial Location (Infratentorial, supratentorial)',
    'EOR (GTR, NTR, STR, biopsy)',
    'WHO Grade'
]


# 2) Keep only sensitive fields present in REAL and ALL SYNTHETIC datasets
valid_sens = []
for s in ALT_SENSITIVE_CANDIDATES:
    if s in real_df.columns and all(s in sdf.columns for sdf in synthetic_datasets.values()):
        valid_sens.append(s)

if not valid_sens:
    print("⚠️ None of the ALT_SENSITIVE_CANDIDATES are available across real+synthetic. Add one that exists in all datasets.")
else:
    # 3) Run attackers per sensitive field
    from sdmetrics.single_table import CategoricalNB, CategoricalKNN, CategoricalRF
    ATTACKERS = {
        'Naive Bayes': CategoricalNB,
        'KNN': CategoricalKNN,
        'Random Forest': CategoricalRF
    }

    alt_rows = []
    for s in valid_sens:
        SENS_LIST = [s]
        for model_name, synth_df in synthetic_datasets.items():
            for atk_name, atk_cls in ATTACKERS.items():
                try:
                    score = atk_cls.compute(
                        real_data=real_df,
                        synthetic_data=synth_df,
                        key_fields=key_fields,
                        sensitive_fields=SENS_LIST
                    )
                except Exception:
                    score = np.nan
                alt_rows.append({
                    'Sensitive': s,
                    'Model': model_name,
                    'Attacker': atk_name,
                    'Privacy Score (↑ safer)': score
                })

    alt_attack_df = pd.DataFrame(alt_rows).sort_values(['Sensitive','Attacker','Model']).reset_index(drop=True)
    print("✅ Alternate-sensitive attacker results:")
    try:
        display(alt_attack_df)
    except NameError:
        print(alt_attack_df.head())

    # 4) Poster-ready heatmap per sensitive field (RdYlGn: low=red (riskier), high=green (safer))
    MODEL_ORDER = ['GaussianCopula','CopulaGAN','CTGAN','TVAE']
    ATTACKER_ORDER = ['Naive Bayes','KNN','Random Forest']

    for s in valid_sens:
        sub = alt_attack_df[alt_attack_df['Sensitive'] == s]
        if sub.empty:
            continue
        pivot_df = sub.pivot(index='Model', columns='Attacker', values='Privacy Score (↑ safer)')
        # enforce consistent ordering where present
        pivot_df = pivot_df.reindex(index=[m for m in MODEL_ORDER if m in pivot_df.index],
                                    columns=[a for a in ATTACKER_ORDER if a in pivot_df.columns])

        plt.figure(figsize=(8, 4), dpi=300)
        img = plt.imshow(pivot_df.values, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn', interpolation='nearest')
        plt.title(f'Attribute-Inference Privacy (Sensitive = "{s}")\nLower (red) = Higher attack success; Higher (green) = Safer', fontsize=11)
        plt.xticks(range(pivot_df.shape[1]), list(pivot_df.columns), rotation=0)
        plt.yticks(range(pivot_df.shape[0]), list(pivot_df.index))

        # Annotate cells
        for i in range(pivot_df.shape[0]):
            for j in range(pivot_df.shape[1]):
                val = pivot_df.values[i, j]
                if not np.isnan(val):
                    plt.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=9)

        cbar = plt.colorbar(img)
        cbar.set_label('Privacy Score (↑ safer, 1.0 best)', rotation=90)
        plt.tight_layout()
        outpath = f'../legacy/original_submission/figures/privacy_heatmap_sensitive_{s.replace(" ","_").replace("/","-")}_withanon.png'
        plt.savefig(outpath, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"📁 Saved: {outpath}")

    # 5) Combined single-panel heatmap (columns = Sensitive–Attacker pairs)
    alt_attack_df['Combo'] = alt_attack_df['Sensitive'] + ' – ' + alt_attack_df['Attacker']
    combo = alt_attack_df.pivot(index='Model', columns='Combo', values='Privacy Score (↑ safer)')

    # Order columns by sensitive then attacker
    combos_order = []
    for s in valid_sens:
        for a in ATTACKER_ORDER:
            col = f"{s} – {a}"
            if col in combo.columns:
                combos_order.append(col)

    combo = combo.reindex(index=[m for m in MODEL_ORDER if m in combo.index],
                          columns=combos_order)

    if combo.size > 0:
        plt.figure(figsize=(min(14, 3 + 1.2*max(4,len(combo.columns))), 5), dpi=300)
        img = plt.imshow(combo.values, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn', interpolation='nearest')
        plt.title('Simulated Attribute-Inference Privacy by Sensitive Variable and Attacker\nLower (red) = Higher attack success; Higher (green) = Safer', fontsize=11)
        plt.xticks(range(combo.shape[1]), list(combo.columns), rotation=45, ha='right')
        plt.yticks(range(combo.shape[0]), list(combo.index))

        for i in range(combo.shape[0]):
            for j in range(combo.shape[1]):
                val = combo.values[i, j]
                if not np.isnan(val):
                    plt.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8)

        cbar = plt.colorbar(img)
        cbar.set_label('Privacy Score (↑ safer, 1.0 best)', rotation=90)
        plt.tight_layout()
        outpath = '../legacy/original_submission/figures/privacy_heatmap_multi_sensitive_withanon.png'
        plt.savefig(outpath, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"📁 Saved: {outpath}")

    # 6) Optional: small legend banner explaining color semantics, saved as its own PNG for your poster
    plt.figure(figsize=(4, 0.6), dpi=300)
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    plt.imshow(grad, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    plt.gca().set_axis_off()
    plt.title('Privacy Score:   0 = Higher Attack Success (risk)   →   1 = Safer', fontsize=9, pad=6)
    legend_path = '../legacy/original_submission/figures/privacy_colormap_legend_withanon.png'
    plt.savefig(legend_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"📁 Saved: {legend_path}")


In [ ]:
# --- Shorthand mappings ---
short_sensitive = {
    'MRN': 'MRN',
    'Initials': 'Initials',
    'Date of Birth': 'DOB',
    'Date of Sx': 'DOS',
    'Death (Y/N)': 'Death',
    'Recurrence': 'Recurrence',
    'Postop complication': 'Postop Comp',
    'Age at Diagnosis': 'Age',
    'BMI': 'BMI',
    'Cranial Location (Infratentorial, supratentorial)': 'Cranial Location',
    'EOR (GTR, NTR, STR, biopsy)': 'EOR',
    'WHO Grade': 'WHO'
}

short_attacker = {
    'Naive Bayes': 'NB',
    'Random Forest': 'RF',
    'KNN': 'KNN'
}

# Apply renaming for simpler column labels
alt_attack_df['Short Sensitive'] = alt_attack_df['Sensitive'].map(short_sensitive)
alt_attack_df['Short Attacker'] = alt_attack_df['Attacker'].map(short_attacker)
alt_attack_df['Combo'] = alt_attack_df['Short Sensitive'] + ' – ' + alt_attack_df['Short Attacker']

combo = alt_attack_df.pivot(index='Model', columns='Combo', values='Privacy Score (↑ safer)')
combo = combo.reindex(index=MODEL_ORDER)

if combo.size > 0:
    fig, ax = plt.subplots(figsize=(15, 3), dpi=300)

    img = ax.imshow(combo.values, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn', interpolation='nearest')

    # Titles and labels
    ax.set_title('Simulated Attribute-Inference Privacy\nShort Labels (↑ Safer)', fontsize=12)
    ax.set_xticks(np.arange(combo.shape[1]))
    ax.set_xticklabels(combo.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(np.arange(combo.shape[0]))
    ax.set_yticklabels(combo.index, fontsize=10, fontweight='bold')  # Models

    # Annotate scores
    for i in range(combo.shape[0]):
        for j in range(combo.shape[1]):
            val = combo.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=7)

    # Colorbar
    cbar = fig.colorbar(img, ax=ax)
    cbar.set_label('Privacy Score (↑ safer)', rotation=90)

    plt.tight_layout()
    plt.savefig('../legacy/original_submission/figures/final_anon_full_heatmap.png', dpi=300, bbox_inches='tight')


In [ ]:
# --- Create risk_df from alt_attack_df ---
risk_df = alt_attack_df.copy()
risk_df['Risk (↓ better)'] = (1.0 - risk_df['Privacy Score (↑ safer)']) * 100.0

# --- Risk strip plot per model (all pairs) ---
import numpy as np
import matplotlib.pyplot as plt

models = ['GaussianCopula','CopulaGAN','CTGAN','TVAE']
risk_pts = []

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(risk_df)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline

# ---- Settings ----
synthetic_df = synthetic_datasets['CopulaGAN']  # Use any model: CTGAN, TVAE, etc.
key_fields = [
    'Age at Diagnosis', 'Sex', 'BMI',
    'Cranial Location (Infratentorial, supratentorial)',
    'WHO Grade', 'Age at Sx'
]

# Define sensitive fields
CATEGORICAL_SENSITIVE = [
    'Death (Y/N)', 'Recurrence', 'Postop complication',
    'Cranial Location (Infratentorial, supratentorial)',
    'EOR (GTR, NTR, STR, biopsy)', 'WHO Grade'
]
NUMERICAL_SENSITIVE = ['Age at Diagnosis', 'BMI']
ALL_SENSITIVE = CATEGORICAL_SENSITIVE + NUMERICAL_SENSITIVE

# --- Create storage
predictions = pd.DataFrame(index=real_df.index)
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# --- Categorical Fields ---
for s in CATEGORICAL_SENSITIVE:
    try:
        train_df = synthetic_df[key_fields + [s]].dropna()
        test_df = real_df[key_fields + [s]].dropna()

        X_train = train_df[key_fields]
        y_train = train_df[s].astype(str)

        X_test = test_df[key_fields]
        y_test = test_df[s].astype(str)

        pipe = Pipeline([
            ('encoder', encoder),
            ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
        ])

        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        predictions.loc[y_test.index, s] = preds
    except Exception as e:
        print(f"⚠️ Failed categorical {s}: {e}")
        predictions[s] = np.nan

# --- Numerical (binned into 4 quartiles as strings) ---
for s in NUMERICAL_SENSITIVE:
    try:
        train_df = synthetic_df[key_fields + [s]].dropna()
        test_df = real_df[key_fields + [s]].dropna()

        # Bin to quartiles as strings
        train_df[s] = pd.qcut(train_df[s], q=4, duplicates='drop').astype(str)
        test_df[s] = pd.qcut(test_df[s], q=4, duplicates='drop').astype(str)

        X_train = train_df[key_fields]
        y_train = train_df[s]

        X_test = test_df[key_fields]
        y_test = test_df[s]

        pipe = Pipeline([
            ('encoder', encoder),
            ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
        ])

        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        predictions.loc[y_test.index, s] = preds
    except Exception as e:
        print(f"⚠️ Failed numerical {s}: {e}")
        predictions[s] = np.nan

# --- Align and compare ---
truth = real_df[ALL_SENSITIVE].astype(str).reindex_like(predictions)

# Boolean matrix of correct predictions
correct_matrix = (predictions == truth)

# Joint re-identification
joint_match = correct_matrix.all(axis=1)
partial_match = correct_matrix.any(axis=1) & ~joint_match
no_match = ~correct_matrix.any(axis=1)

# --- Summary ---
n_total = len(correct_matrix)
n_joint = joint_match.sum()
n_partial = partial_match.sum()
n_none = no_match.sum()

print(f"\n🔍 Joint Inference Results:")
print(f"✅ Exact Match (joint re-identification): {n_joint} / {n_total} patients ({n_joint/n_total*100:.1f}%)")
print(f"⚠️ Partial Match (1+ correct fields): {n_partial} / {n_total} patients ({n_partial/n_total*100:.1f}%)")
print(f"❌ No Match: {n_none} / {n_total} patients ({n_none/n_total*100:.1f}%)")

# Optional: export or visualize
joint_df = pd.DataFrame({
    'Exact Match': joint_match,
    'Partial Match': partial_match,
    'No Match': no_match
}, index=real_df.index)

display(joint_df)



In [ ]:
# Number of sensitive fields correctly inferred per patient
leak_count_per_patient = correct_matrix.sum(axis=1)

leak_count_df = pd.DataFrame({
    'Patient Index': correct_matrix.index,
    'Number of Leaked Fields': leak_count_per_patient
})

display(leak_count_df)


In [ ]:
import matplotlib.pyplot as plt

# Bin leakage counts
bins = leak_count_per_patient.value_counts().sort_index()

plt.figure(figsize=(6, 4), dpi=300)
plt.bar(bins.index.astype(str), bins.values, color="#4C72B0", alpha=0.85)

plt.xlabel("Number of Sensitive Fields Correctly Inferred")
plt.ylabel("Number of Patients")
plt.title("Distribution of Attribute Leakage per Patient")

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
collapsed = leak_count_per_patient.apply(
    lambda x: '0' if x == 0 else ('1' if x == 1 else ('2' if x == 2 else '3+'))
)

collapsed_counts = collapsed.value_counts().sort_index()

plt.figure(figsize=(6, 4), dpi=300)
plt.bar(collapsed_counts.index, collapsed_counts.values, color="#DD8452", alpha=0.85)

plt.xlabel("Number of Leaked Sensitive Attributes")
plt.ylabel("Number of Patients")
plt.title("Per‑Patient Attribute Leakage (Collapsed Categories)")

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# --- Risk strip plot per model (all pairs) ---
import numpy as np
import matplotlib.pyplot as plt

models = ['GaussianCopula','CopulaGAN','CTGAN','TVAE']
risk_pts = []
for m in models:
    vals = risk_df.loc[risk_df['Model']==m, 'Risk (↓ better)'].dropna().values
    risk_pts.append(vals)

plt.figure(figsize=(8,5), dpi=300)
for idx, vals in enumerate(risk_pts):
    x = np.random.normal(loc=idx, scale=0.05, size=len(vals))  # jitter
    plt.scatter(x, vals, s=18, alpha=0.7)
    if len(vals):
        med = np.median(vals)
        plt.plot([idx-0.25, idx+0.25], [med, med], linewidth=3)

plt.xticks(range(len(models)), models)
plt.ylabel('Risk = 1 − Privacy Score (↓ better)')
plt.title('Risk Distribution Across Sensitive×Attacker Pairs')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../legacy/original_submission/figures/risk_stripplot_withanon.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# --- Risk strip plot per model with attacker-specific markers (consistent coloring) ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

models    = ['GaussianCopula','CopulaGAN','CTGAN','TVAE']
attackers = ['Naive Bayes','KNN','Random Forest']

marker_map = {
    'Naive Bayes': '^',   # triangle
    'KNN': 'x',           # x
    'Random Forest': 's'  # square
}

offset_map = {
    'Naive Bayes': -0.15,
    'KNN': 0.0,
    'Random Forest': 0.15
}

plt.figure(figsize=(8, 5), dpi=300)

jitter_scale = 0.03
point_size   = 36

for mi, m in enumerate(models):
    sub_m = risk_df[risk_df['Model'] == m]
    for atk in attackers:
        vals = sub_m.loc[sub_m['Attacker'] == atk, 'Risk (↓ better)'].dropna().values
        if len(vals) == 0:
            continue
        center = mi + offset_map[atk]
        x = np.random.normal(loc=center, scale=jitter_scale, size=len(vals))
        mk = marker_map[atk]
        # same fill/edge style for all markers
        plt.scatter(
            x, vals,
            marker=mk,
            s=point_size,
            alpha=0.8,
            facecolors='lightgray',
            edgecolors='black',
            linewidths=0.8
        )

plt.xticks(range(len(models)), models)
plt.ylabel('Attack Success Score (↑ = more vulnerable)')
plt.title('Attack Success Risk by Model (Attacker = marker shape)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

legend_handles = [
    Line2D([0], [0], marker=marker_map[a], color='black',
           markerfacecolor='lightgray', markeredgecolor='black',
           markersize=7, linestyle='None', label=a)
    for a in attackers
]
plt.legend(handles=legend_handles, title='Attacker', loc='upper right', frameon=True)

plt.tight_layout()
plt.savefig('../legacy/original_submission/figures/risk_stripplot_markers_colored_withanon.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
risk_df

In [ ]:
TREATMENT = 'EOR (GTR, NTR, STR, biopsy)'

OUTCOME_VARIABLES = [
    'PFS (After Sx; Months)',
    'Overall Survival (yrs)',
    'Temporary EVD required', 'Shunt placed preop',
    'Shunt placed postop', 'Postop Hydrocephalus', 'New/worse Postop CN Palsy Y/N',
    'Postop Weakness', 'Postop Sensory Changes', 'Postop cerebellar deficit',
    'Postop complication', 'Followup CN Palsy Y/N',
    'Followup Weakness', 'Followup Sensory Changes',
    'Cerebellar function at followup compared to preop', 'Vertigo at followup', 'Intraop Complications', 
    'Recurrence'
]

COVARIATES = [
    'Age at Diagnosis', 'Age at Sx', 'BMI',
    'Tumor Size 1 (cm)', 'Tumor Size 2 (cm)', 'Tumor Size 3 (cm)',
    'Duration of symptoms preop (months)', 'Sex', 'Cyst?', 'Recurrent Tumor (Y/N)','Ki-67_index_7%', 
    'Prior surgery', 'Prior Chemo', 'Prior radiation',
    'TERT_marker', 'Synaptophysin_marker', 'GFAP_marker', 'Olig2_marker',
    'Preop Hydrocephalus', 'Preop CN Palsy (Y/N)', 'Preop Papilledema (Y/N)', 'Preop HA',
    'Preop Nausea/Vomiting', 'Preop vertigo', 'Preop Cerebellar Signs', 'Preop Weakness',
    'Preop Sensory Loss', 'Cranial Location (Infratentorial, supratentorial)',
    'Intra vs Extra Ventricular vs Both', 'Laterality',
    'WHO Grade', 'Enhancement',  'Adjuvant therapy (Y/N)', 'Adjuvant chemo', 'Radiotherapy (Y/N)'
]


In [ ]:
real_df['Recurrence'].value_counts()

In [ ]:
synthetic_df['Recurrence'].value_counts()

In [ ]:
numerical_outcomes = ['PFS (After Sx; Months)', 'Overall Survival (yrs)']
categorical_outcomes = [col for col in OUTCOME_VARIABLES if col not in numerical_outcomes]

encoded_df = synthetic_df.copy()

# Encode only categorical outcome vars
for col in categorical_outcomes:
    encoded_df[col] = encoded_df[col].astype('category').cat.codes

# Keep numerical as-is
X = encoded_df[OUTCOME_VARIABLES]  # already mixed numerical + encoded binary

# Fit GMM
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3, random_state=42)
latent_labels = gmm.fit_predict(X)
encoded_df['Latent_Class'] = latent_labels

# Profiles
class_profiles = encoded_df.groupby('Latent_Class')[OUTCOME_VARIABLES].mean()


# Step 3: Plot heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.heatmap(class_profiles, annot=True, cmap="Blues")
plt.title("Outcome Probabilities by Latent Class")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.countplot(x='Latent_Class', data=encoded_df, palette='Set2')
plt.title("Count of Patients per Latent Class")
plt.xlabel("Latent Class")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()
